In [ ]:
#| default_exp latechunk_eval
#| eval: false

# Late Chunking Accuracy Eval

In [ ]:
#| eval: false
import os, certifi
os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())
if os.environ.get('LITESEARCH_INSECURE_SSL') == '1':   # opt-in only; corporate MITM proxy fallback
    import requests as _r, urllib3 as _u
    _u.disable_warnings()
    _orig_send = _r.Session.send
    _r.Session.send = lambda self, *a, **k: _orig_send(self, *a, **{**k, 'verify': False})

from litesearch import database
from litesearch.utils import FastEncode, AutoLateChunkFastEncode, nomic_text_v15
from litesearch.data import chunk_spans
import numpy as np, json
from datasets import load_dataset

In [ ]:
#| eval: false
_probe = load_dataset('dwzhu/LongEmbed', 'narrativeqa')
print(_probe)
for split in _probe: print(split, _probe[split].column_names, dict(list(_probe[split][0].items())[:3]).keys())

In [ ]:
#| eval: false
TASKS = ['narrativeqa', '2wikimqa']

def load_longembed(task, max_docs=200, max_queries=100):
    'Load a LongEmbed task capped to max_docs/max_queries; keep only queries whose judged doc survives the cap.'
    ds = load_dataset('dwzhu/LongEmbed', task)
    corpus = {r['doc_id']: r['text'] for r in ds['corpus'].select(range(min(max_docs, len(ds['corpus']))))}
    queries = {r['qid']: r['text'] for r in ds['queries']}
    qrels = {}
    for r in ds['qrels']:
        qid, did = r['qid'], r['doc_id']
        if did in corpus and qid in queries: qrels.setdefault(qid, set()).add(did)
    queries = {q: queries[q] for q in list(qrels)[:max_queries]}
    qrels = {q: qrels[q] for q in queries}
    return corpus, queries, qrels

In [ ]:
#| eval: false
_corpus,_queries,_qrels = load_longembed('narrativeqa', max_docs=20, max_queries=5)
assert isinstance(_corpus, dict) and isinstance(_queries, dict) and isinstance(_qrels, dict)
assert len(_corpus) <= 20 and len(_queries) <= 5
for qid, docs in _qrels.items():
    assert qid in _queries
    assert all(d in _corpus for d in docs)
assert len(_qrels) >= 1

In [ ]:
#| eval: false
def agg_docs(hits):
    'Collapse ranked chunk hits to unique parent doc_ids, keeping best rank.'
    seen, out = set(), []
    for h in hits:
        d = json.loads(h['metadata'])['doc_id']
        if d not in seen: seen.add(d); out.append(d)
    return out

def recall_at_k(ranked, relevant, k):
    'Fraction of relevant docs present in the top-k ranking.'
    if not relevant: return 0.0
    return len(set(ranked[:k]) & relevant) / len(relevant)

def ndcg_at_k(ranked, relevant, k=10):
    'Binary-relevance nDCG@k.'
    dcg = sum(1.0/np.log2(i+2) for i,d in enumerate(ranked[:k]) if d in relevant)
    idcg = sum(1.0/np.log2(i+2) for i in range(min(len(relevant), k)))
    return float(dcg/idcg) if idcg else 0.0

In [ ]:
#| eval: false
_hits = [{'metadata': json.dumps({'doc_id':'A','chunk_idx':0})},
         {'metadata': json.dumps({'doc_id':'B','chunk_idx':3})},
         {'metadata': json.dumps({'doc_id':'A','chunk_idx':1})}]
assert agg_docs(_hits) == ['A','B']                     # dedup, keep first occurrence
assert recall_at_k(['A','B','C'], {'A','C'}, 3) == 1.0
assert recall_at_k(['A','B','C'], {'A','C'}, 1) == 0.5
assert abs(ndcg_at_k(['A','B'], {'A'}, 10) - 1.0) < 1e-9   # relevant doc first -> perfect
assert ndcg_at_k(['B','A'], {'A'}, 10) < 1.0               # relevant doc second -> discounted

In [ ]:
#| eval: false
import types

def _store(corpus, rows):
    'Build an in-memory litesearch store from prepared rows; patches search to always return metadata.'
    db = database()
    st = db.get_store()
    st.insert_all(rows)
    _orig = db.search.__func__
    def _search(self, q, emb, columns=None, **kw):
        cols = list(columns) if columns else []
        if 'metadata' not in cols: cols = ['metadata'] + cols
        return _orig(self, q, emb, columns=cols, **kw)
    db.search = types.MethodType(_search, db)
    return db

def build_naive(corpus, enc):
    'Chunk-then-embed each chunk independently.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt)
        embs = enc.encode_document([t for _,_,t in spans])
        for ci,((_,_,t),e) in enumerate(zip(spans,embs)):
            rows.append({'content':t,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':ci})})
    return _store(corpus, rows)

def build_fulldoc(corpus, enc):
    'One embedding per whole document.'
    rows = []
    for did,txt in corpus.items():
        e = enc.encode_document([txt])[0]
        rows.append({'content':txt,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':0})})
    return _store(corpus, rows)

def build_latechunk(corpus, lc_enc):
    'Late chunking: embed whole doc, pool per chunk span.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt)
        embs,_ = lc_enc.encode_auto(txt, [(s,e) for s,e,_ in spans])
        for ci,((_,_,t),e) in enumerate(zip(spans,embs)):
            rows.append({'content':t,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':ci})})
    return _store(corpus, rows)

_SIT_CACHE = {}
def situate(chat, doc, chunk):
    'Generate (and cache) a short doc-situating blurb for a chunk via local Gemma.'
    key = (hash(doc), hash(chunk))
    if key in _SIT_CACHE: return _SIT_CACHE[key]
    from rishi.core import resp_text
    prompt = (f"<document>\n{doc}\n</document>\nHere is a chunk from it:\n<chunk>\n{chunk}\n</chunk>\n"
              "Give a short sentence situating this chunk within the document for search. Answer with only that sentence.")
    blurb = resp_text(chat(prompt)).strip()
    _SIT_CACHE[key] = blurb
    return blurb

def build_contextual(corpus, enc, chat):
    'Contextual retrieval: prepend a Gemma-generated situating blurb to each chunk before embedding.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt)
        texts = [f"{situate(chat, txt, t)}\n{t}" for _,_,t in spans]
        embs = enc.encode_document(texts)
        for ci,(t,e) in enumerate(zip(texts,embs)):
            rows.append({'content':t,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':ci})})
    return _store(corpus, rows)

In [ ]:
#| eval: false
_corpus = {'d1':'Cats are small mammals kept as pets. They purr when content.',
           'd2':'The Eiffel Tower is in Paris. It was built in 1889 for the World Fair.'}
_enc = FastEncode(model_dict=nomic_text_v15, max_seq_len=2048)
_lc  = AutoLateChunkFastEncode(model_dict=nomic_text_v15, max_seq_len=2048)
for _db in (build_naive(_corpus,_enc), build_fulldoc(_corpus,_enc), build_latechunk(_corpus,_lc)):
    _q = 'where is the eiffel tower'
    _hits = _db.search(_q, _enc.encode_query(_q).tobytes(), limit=10)
    assert agg_docs(_hits)[0] == 'd2'          # obvious relevant doc ranks first